## Creates matched_phrases table based on deprel 

### base table for later filtering since it contains column for semantic role


In [1]:
import sqlite3
import pandas as pd

## Configuration

In [2]:
# verbimustrite andmebaas
PATTERN_DB = "../example_data/verb_patterns.db"

# transaktsioonide andmebaas
TRANSACTIONS_DB = "../example_data/transactions.db"

# enriched transaktsioonide andmebaas
ENRICHED_TRANSACTIONS_DB = "../example_data/enriched_transactions.db"

# Siia salvestuvad loodavad tabelid
PATTERN_MATCHES_DB = "../example_data/pattern_matches.db"

ENRICHED_TRANSACTIONS_TABLE = "transaction_v2"
PATTERNS_TABLE = "patterns"
VERB_MATCHES = "verb_matches"
SEMANTIC_ANN = "semantic_annotations"

# new patterns table with semantic role
PATTERNS_WITH_ROLE = "patterns_role"

# temporary table for pattern-join-transaction_head
PATTERNS_HEAD = "patterns_tr_head"

# result table
MATCHES_TABEL = "matched_phrases"

# role condition
# can be included for the final table
ROLE_CONDITION = 'isik_alati'

## Connect to db

In [3]:
con = sqlite3.connect(PATTERN_MATCHES_DB)
cur = con.cursor()

# transaktsioonide andmebaasi lisamine
cur.execute(f'ATTACH DATABASE "{TRANSACTIONS_DB}" AS trans')

# transaktsioonide andmebaasi lisamine
cur.execute(f'ATTACH DATABASE "{ENRICHED_TRANSACTIONS_DB}" AS entrans')

# pattern_matches andmebaasi lisamine
cur.execute(f'ATTACH DATABASE "{PATTERN_DB}" AS pat')

## Workflow



### Add semantic role to patterns

In [11]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=PATTERNS_WITH_ROLE))

cur.execute("""
CREATE TABLE {tbl1} AS
SELECT DISTINCT
    pat.pat_id as pat_id,
    pat.verb_word as verb_word,
    pat.verb_compound as verb_compound,
    pat.phrase_nr as phrase_nr,
    pat.phrase_case as phrase_case,
    pat.deprel as deprel,
    semantic_role ||'_'||certainty as role
FROM 
    {pattbl} as pat
INNER JOIN 
    {semann} as ann
ON
    pat.pat_id = ann.pattern_id

""".format(tbl1=PATTERNS_WITH_ROLE, pattbl=PATTERNS_TABLE, semann=SEMANTIC_ANN))

CPU times: user 50.7 ms, sys: 7.19 ms, total: 57.9 ms
Wall time: 69.6 ms


### temp table for join

In [16]:
%%time

cur.execute("""
DROP TABLE IF EXISTS {tbl}
""".format(tbl=PATTERNS_HEAD))

cur.execute("""
CREATE TABLE {tbl1} AS
SELECT DISTINCT
    head.id as head_id,
    pat.pat_id as pat_id,
    pat.verb_word as verb_word,
    pat.verb_compound as verb_compound,
    pat.phrase_nr as phrase_nr,
    pat.phrase_case as phrase_case,
    pat.deprel as pat_deprel,
    pat.role as role
FROM 
    {pattbl} as pat
INNER JOIN 
    trans.transaction_head as head
ON
    pat.verb_word = head.verb

""".format(tbl1=PATTERNS_HEAD, pattbl=PATTERNS_WITH_ROLE))

CPU times: user 74.7 ms, sys: 4.12 ms, total: 78.8 ms
Wall time: 82.7 ms


### matched_phrases

Praegu on joini aluseks head_id, deprel, feats (vajadusel ka semantic_role). 

` NB!!! patterns tabeli phrase_case on abl/all/ad jne ja neid tulebks matchida feats veerus olevaga`

In [5]:
%%time 

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=MATCHES_TABEL))


cur.execute("""
CREATE TABLE {new_table} AS
SELECT DISTINCT
    tbl1.head_id as head_id,
    tbl1.pat_id as pat_id,
    tr.id as transaction_id,
    tbl1.phrase_nr as phrase_nr,
    tbl1.verb_word as verb_word,
    tbl1.verb_compound as verb_compound,
    tr.lemma as root_word,
    tbl1.pat_deprel as deprel,
    tbl1.phrase_case as phrase_case,
    tbl1.role as semantic_role,
    tr.koht as koht,
    elus as elus
    
FROM 
    {pattbl} as tbl1
JOIN 
    entrans.{trans_tbl} as tr
ON 
    tbl1.head_id = tr.head_id
    and tbl1.pat_deprel = tr.deprel
WHERE INSTR(',' || tr.feats || ',', ',' || tbl1.phrase_case || ',') > 0
-- and role = {role}
""".format(new_table=MATCHES_TABEL, pattbl=PATTERNS_HEAD, trans_tbl=ENRICHED_TRANSACTIONS_TABLE))

CPU times: user 28.1 ms, sys: 2.97 ms, total: 31.1 ms
Wall time: 37.1 ms


### kontroll

In [6]:
query = """SELECT * FROM sqlite_master WHERE type='table'"""
source = pd.read_sql_query(query, con)
source

,type,name,tbl_name,rootpage,sql
0,table,patterns_role,patterns_role,2,"CREATE TABLE patterns_role(\n pat_id INT,\n ..."
1,table,patterns_tr_head,patterns_tr_head,335,"CREATE TABLE patterns_tr_head(\n head_id INT,..."
2,table,matched_phrases,matched_phrases,603,"CREATE TABLE matched_phrases(\n head_id INT,\..."


In [8]:
query = """SELECT * from {tbl} limit 10""".format(tbl=MATCHES_TABEL)
source = pd.read_sql_query(query, con)
source

,head_id,pat_id,transaction_id,phrase_nr,verb_word,verb_compound,root_word,deprel,phrase_case,semantic_role,koht,elus
0,179,4,258,1,nõudma,,mina,obl,abl,isik_alati,UNK,YES
1,179,4,258,1,nõudma,,mina,obl,abl,koht_mitte kunagi,UNK,YES
2,179,4,258,1,nõudma,,mina,obl,abl,muu_mitte kunagi,UNK,YES
3,179,134,258,1,nõudma,tagasi,mina,obl,abl,isik_alati,UNK,YES
4,179,134,258,1,nõudma,tagasi,mina,obl,abl,koht_mitte kunagi,UNK,YES
5,179,134,258,1,nõudma,tagasi,mina,obl,abl,muu_mitte kunagi,UNK,YES
6,179,154,258,1,nõudma,välja,mina,obl,abl,isik_alati,UNK,YES
7,179,154,258,1,nõudma,välja,mina,obl,abl,koht_mitte kunagi,UNK,YES
8,179,154,258,1,nõudma,välja,mina,obl,abl,muu_mitte kunagi,UNK,YES
9,179,610,258,1,nõudma,sisse,mina,obl,abl,isik_alati,UNK,YES


In [9]:
con.close()